In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Ayarlar
INFANT_DATA_PATH = r"C:\Users\Dogan\Desktop\Ayberk Dosyalar\Bitirme Projesi\Fundus\all_fundus_datas\vessel_segment_bebek_data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2  # EfficientNet-B5 yüksek VRAM tüketir, 2 güvenlidir
IMG_SIZE = (512, 512) 
LEARNING_RATE = 1e-5 # Fine-tuning için düşük hız (önceki bilgileri korumak için)
EPOCHS = 100



class InfantFundusDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.img_dir = os.path.join(root_dir, split, "image")
        self.mask_dir = os.path.join(root_dir, split, "mask")
        
        valid_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.bmp')
        self.image_names = [f for f in os.listdir(self.img_dir) 
                            if f.lower().endswith(valid_extensions)]
        
        self.transform = transform
        print(f"--> {split} seti için {len(self.image_names)} adet görsel bulundu.")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Dosya adını uzantısız al (örn: "017_fundus")
        base_name = os.path.splitext(img_name)[0]
        
        # Maskeyi bulmak için farklı uzantıları dene
        mask_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.tif', '.png']:
            temp_path = os.path.join(self.mask_dir, base_name + ext)
            if os.path.exists(temp_path):
                mask_path = temp_path
                break
        
        # Görüntü ve Maskeyi oku
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if mask_path else None

        if image is None:
            raise FileNotFoundError(f"Görüntü okunamadı: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Maske bulunamadı veya okunamadı! Aranan isim: {base_name} (Klasör: {self.mask_dir})")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask




train_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_ds = InfantFundusDataset(INFANT_DATA_PATH, split="train", transform=train_transform)
val_ds = InfantFundusDataset(INFANT_DATA_PATH, split="val", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)



# 1. Modeli 3 sınıf (BG, Arter, Ven) için kur
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights=None, 
    in_channels=3,
    classes=3, 
).to(DEVICE)

# 2. Yetişkin modelinden ağırlıkları yükle
pretrained_path = "vessel_yetiskin_model.pth"
if os.path.exists(pretrained_path):
    checkpoint = torch.load(pretrained_path)
    model_dict = model.state_dict()
    
    # Sadece boyutu uyan katmanları al (Segmentation Head hariç her yer)
    filtered_dict = {k: v for k, v in checkpoint.items() if k in model_dict and v.size() == model_dict[k].size()}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)
    print(f"--> {len(filtered_dict)} katman başarıyla aktarıldı.")
else:
    print("!! Önceden eğitilmiş model bulunamadı, sıfırdan başlanıyor.")

criterion = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


# --- 5. HÜCRE GÜNCELLEME ---
best_iou = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(DEVICE), masks.to(DEVICE).long()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE).long()
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            stats = smp.metrics.get_stats(preds, masks, mode='multiclass', num_classes=3)
            tp += stats[0]; fp += stats[1]; fn += stats[2]; tn += stats[3]

    current_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro-imagewise").item()
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val IoU: {current_iou:.4f}")

    if current_iou > best_iou:
        best_iou = current_iou
        torch.save(model.state_dict(), "best_bebek_infant_model2.pth")
        print("--> Bebek modeli kaydedildi!")